# Projet AirBnB -- Prediction de Prix par IA Supervisee

**Cours Python - Analyse de Donnees - M. Sayf BEJAOUI**

**Objectif :** Predire le prix optimal d'un logement AirBnB a partir de ses caracteristiques.

**Pipeline :** Collecte -> Pretraitement -> Analyse -> Feature Engineering -> Modelisation -> Evaluation -> Prediction

**Donnees :** Lyon, Paris, Bordeaux -- Juin 2025 (79 colonnes, insideairbnb.com)

## 1. Imports

On importe les modules `src/` du projet :
- `src.data.collect` -- telechargement des donnees
- `src.data.preprocess` -- nettoyage (4 etapes)
- `src.features.build_features` -- encodage et outliers
- `src.models.train` -- entrainement et evaluation

`sys.path.insert(0, '..')` permet d'acceder a `src/` depuis le dossier `notebook/`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

# Ajoute la racine du projet au chemin Python pour importer src/
sys.path.insert(0, os.path.abspath('..'))

# --- Modules du projet src/ ---
from src.data.collect import download_all
from src.data.preprocess import preprocess, save_processed
from src.features.build_features import build_features, FEATURE_COLS_SIMPLE, FEATURE_COLS_MULTIPLE
from src.models.train import run_pipeline

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CITIES = {'Lyon': 'lyon', 'Paris': 'paris', 'Bordeaux': 'bordeaux'}
COLORS = {'Lyon': 'steelblue', 'Paris': 'coral', 'Bordeaux': 'mediumseagreen'}
os.makedirs('../reports/figures', exist_ok=True)
print('Imports OK -- modules src/ charges')

## 2. Collecte des Donnees

`download_all()` est definie dans `src/data/collect.py`.
Elle telecharge les `.csv.gz` depuis insideairbnb.com et les decompresse.
Affiche `[SKIP]` si le fichier existe deja.

| Ville | Date | Taille |
|-------|------|--------|
| Lyon | 15/06/2025 | 18.9 Mo |
| Paris | 06/06/2025 | 173 Mo |
| Bordeaux | 15/06/2025 | 25 Mo |

In [ ]:
# Telecharge et decompresse les 3 fichiers (ignore si deja present)
download_all(raw_dir='../data/raw')

In [ ]:
for name, key in CITIES.items():
    df = pd.read_csv(f'../data/raw/{key}/listings_detail.csv', low_memory=False)
    print(f'{name:10s} : {df.shape[0]:,} annonces x {df.shape[1]} colonnes')

## 3. Pretraitement des Donnees

`preprocess()` est definie dans `src/data/preprocess.py`.

| Etape | Action |
|-------|--------|
| **3a** | 79 -> 26 colonnes pertinentes |
| **3b** | Suppression des doublons (`id`) |
| **3c** | NaN -> 0, lignes sans prix -> supprimees |
| **3d** | `"$120"` -> float, `"t"/"f"` -> 1/0, `"95%"` -> 95.0 |

In [ ]:
# preprocess() = load_raw + select_columns + remove_duplicates + handle_missing + convert_types
dfs = {}
for name, key in CITIES.items():
    dfs[name] = preprocess(key, raw_dir='../data/raw')

In [ ]:
# Sauvegarde dans data/processed/<ville>_clean.csv
for name, key in CITIES.items():
    save_processed(dfs[name], key, out_dir='../data/processed')

## 4. Analyse Descriptive

Comparaison des 3 villes. La **mediane** est preferee a la moyenne
car les prix AirBnB ont une distribution asymetrique (longue queue vers les prix eleves).

In [ ]:
for name, df in dfs.items():
    print(f'\n=== {name} ===')
    display(df[['price','accommodates','bedrooms','beds','bathrooms',
                'review_scores_rating','availability_365']].describe().round(2))

In [ ]:
summary = {}
for name, df in dfs.items():
    summary[name] = {
        'Nb annonces'    : len(df),
        'Prix median (euros)' : round(df['price'].median(), 1),
        'Prix moyen (euros)'  : round(df['price'].mean(), 1),
        'Capacite med.'   : df['accommodates'].median(),
        'Note moyenne'    : round(df['review_scores_rating'].replace(0, float('nan')).mean(), 2),
        'Superhosts (%)'  : round(df['host_is_superhost'].mean() * 100, 1),
    }
display(pd.DataFrame(summary))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, df) in zip(axes, dfs.items()):
    cap = df['price'].quantile(0.99)
    data = df[df['price'] <= cap]['price']
    ax.hist(data, bins=60, color=COLORS[name], edgecolor='white', alpha=0.85)
    ax.axvline(data.median(), color='black', linestyle='--',
               label=f'Mediane : {data.median():.0f} euros')
    ax.set_title(f'Distribution prix - {name}')
    ax.set_xlabel('Prix / nuit (euros)'); ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/prix_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, df) in zip(axes, dfs.items()):
    med = df.groupby('room_type')['price'].median().sort_values()
    ax.barh(med.index, med.values, color=COLORS[name], alpha=0.85)
    ax.set_title(f'Prix par type - {name}')
    ax.set_xlabel('Prix median (euros)')
    for i, v in enumerate(med.values): ax.text(v+1, i, f'{v:.0f} euros', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../reports/figures/prix_par_type.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, (name, df) in zip(axes, dfs.items()):
    top = df.groupby('neighbourhood_cleansed')['price'].median().nlargest(10)
    ax.barh(top.index, top.values, color=COLORS[name], alpha=0.85)
    ax.invert_yaxis()
    ax.set_title(f'Top 10 quartiers - {name}')
    ax.set_xlabel('Prix median (euros)')
plt.tight_layout()
plt.savefig('../reports/figures/top_quartiers.png', bbox_inches='tight')
plt.show()

## 5. Feature Engineering

`build_features()` est definie dans `src/features/build_features.py`.

| Feature | Methode |
|---------|----------|
| `room_type_code` | Encodage ordinal (0=Shared -> 3=Entire) |
| `neighbourhood_freq` | Frequence relative du quartier |
| `property_type_freq` | Frequence relative du type de propriete |

Supprime aussi les outliers de prix (percentiles 1% - 99%).

In [ ]:
# build_features() = encode_room_type + encode_neighbourhood + encode_property_type + remove_price_outliers
for name in list(dfs.keys()):
    print(f'\n{name} :')
    dfs[name] = build_features(dfs[name])
    print(f'  -> {dfs[name].shape[0]:,} annonces x {dfs[name].shape[1]} colonnes')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (name, df) in zip(axes, dfs.items()):
    ax.boxplot(df['price'], vert=False, patch_artist=True,
               boxprops=dict(facecolor=COLORS[name], alpha=0.7))
    ax.set_title(f'Boxplot prix apres nettoyage - {name}')
    ax.set_xlabel('Prix (euros)')
plt.tight_layout()
plt.savefig('../reports/figures/boxplot_prix.png', bbox_inches='tight')
plt.show()

In [ ]:
for name, key in CITIES.items():
    dfs[name].to_csv(f'../data/processed/{key}_features.csv', index=False)
    print(f'{name} -> {dfs[name].shape}')

## 6. Modelisation -- Regression Lineaire

`run_pipeline()` est definie dans `src/models/train.py`.
Elle enchaine : `get_X_y -> split_data (80/20) -> train_linear -> evaluate`.

- **Regression simple** : 1 variable (`accommodates`) -- baseline
- **Regression multiple** : 18 variables -- modele complet

`random_state=42` : resultats reproductibles a chaque execution.

`FEATURE_COLS_SIMPLE` et `FEATURE_COLS_MULTIPLE` sont definis dans `src/features/build_features.py`.

In [ ]:
print('Features simple   :', FEATURE_COLS_SIMPLE)
print('Features multiple :', FEATURE_COLS_MULTIPLE)

results = {}
print('\n=== Entrainement ===')
for name, df in dfs.items():
    print(f'\n{name}  (train={int(len(df)*0.8):,}  test={int(len(df)*0.2):,})')
    # run_pipeline() enchaîne get_X_y + split_data + train_linear + evaluate
    results[f'{name}_simple']   = run_pipeline(df, FEATURE_COLS_SIMPLE,   label='simple')
    results[f'{name}_multiple'] = run_pipeline(df, FEATURE_COLS_MULTIPLE, label='multiple')

### 6a. Regression Lineaire Simple

Variable unique : `accommodates` -- la plus correlee au prix.
Graphique : nuage de points + droite de regression.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, CITIES):
    r = results[f'{name}_simple']
    m, Xte, yte = r['model'], r['X_test'], r['y_test']
    ax.scatter(Xte['accommodates'], yte, alpha=0.25, s=8, color=COLORS[name], label='Reel')
    xs = sorted(Xte['accommodates'].unique())
    ax.plot(xs, m.predict(pd.DataFrame({'accommodates': xs})),
            color='black', linewidth=2, label='Droite')
    ax.set_title(f'Simple - {name} | MAE={r["MAE"]} euros  R2={r["R2"]}')
    ax.set_xlabel('Nb personnes'); ax.set_ylabel('Prix (euros)'); ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/regression_simple.png', bbox_inches='tight')
plt.show()

### 6b. Regression Lineaire Multiple

18 variables : capacite, chambres, scores reviews, hote, disponibilite...
Graphique : prix predit vs prix reel (diagonale = prediction parfaite).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, CITIES):
    r = results[f'{name}_multiple']
    yte, yp = r['y_test'], r['y_pred']
    ax.scatter(yte, yp, alpha=0.2, s=8, color=COLORS[name])
    lims = [min(yte.min(), yp.min()), max(yte.max(), yp.max())]
    ax.plot(lims, lims, 'k--', linewidth=1, label='Prediction parfaite')
    ax.set_title(f'Multiple - {name} | MAE={r["MAE"]} euros  R2={r["R2"]}')
    ax.set_xlabel('Prix reel (euros)'); ax.set_ylabel('Prix predit (euros)'); ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/regression_multiple.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, name in zip(axes, CITIES):
    r = results[f'{name}_multiple']
    coefs = pd.Series(r['model'].coef_, index=r['X_test'].columns).sort_values()
    ax.barh(coefs.index, coefs.values,
            color=['coral' if v < 0 else COLORS[name] for v in coefs])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Coefficients - {name}')
    ax.set_xlabel('Coefficient')
plt.tight_layout()
plt.savefig('../reports/figures/coefficients.png', bbox_inches='tight')
plt.show()

## 7. Evaluation

- **MAE** : erreur moyenne en euros (interpretation directe)
- **RMSE** : penalise davantage les grosses erreurs
- **R2** : proportion de variance expliquee (0 = nul, 1 = parfait)

In [ ]:
rows = []
for name in CITIES:
    for label in ['simple', 'multiple']:
        r = results[f'{name}_{label}']
        rows.append({'Ville': name, 'Modele': label.capitalize(),
                     'MAE (euros)': r['MAE'], 'RMSE (euros)': r['RMSE'],
                     'R2': r['R2'], 'Nb test': len(r['y_test'])})
display(pd.DataFrame(rows))

In [ ]:
# Residus = prix reel - prix predit (ideal : repartis aleatoirement autour de 0)
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
for row, name in enumerate(CITIES):
    for col, label in enumerate(['simple', 'multiple']):
        r   = results[f'{name}_{label}']
        res = r['y_test'].values - r['y_pred']
        ax  = axes[row][col]
        ax.scatter(r['y_pred'], res, alpha=0.25, s=8, color=COLORS[name])
        ax.axhline(0, color='black', linewidth=1, linestyle='--')
        ax.set_title(f'Residus - {name} ({label})')
        ax.set_xlabel('Prix predit (euros)'); ax.set_ylabel('Residu (euros)')
plt.tight_layout()
plt.savefig('../reports/figures/residus.png', bbox_inches='tight')
plt.show()

## 8. Prediction pour un Nouveau Logement

Simulation : **T2, 4 personnes, note 4.8/5, superhost, appartement entier, centre-ville**.

On construit un DataFrame d'une ligne avec les 18 features dans l'ordre exact de l'entrainement,
puis on appelle `model.predict()`.

In [ ]:
new_listing = dict(
    accommodates=4, bedrooms=2, beds=2, bathrooms=1.0,
    minimum_nights=2, availability_365=200,
    number_of_reviews=25, reviews_per_month=1.5,
    review_scores_rating=4.8, review_scores_cleanliness=4.9,
    review_scores_location=4.7, host_is_superhost=1,
    host_response_rate=95.0, calculated_host_listings_count=1,
    instant_bookable=1, room_type_code=3,
    neighbourhood_freq=0.05, property_type_freq=0.4,
)

print('Logement : T2, 4 personnes, note 4.8/5, superhost\n')
for name in CITIES:
    r     = results[f'{name}_multiple']
    feats = list(r['X_test'].columns)  # ordre exact des features a l'entrainement
    X_new = pd.DataFrame([[new_listing.get(f, 0) for f in feats]], columns=feats)
    prix  = round(float(r['model'].predict(X_new)[0]), 2)
    print(f'  {name:10s} -> {prix} euros/nuit')

## Conclusion

| Ville | R2 Simple | R2 Multiple | Prix estime (T2, 4p, superhost) |
|-------|-----------|-------------|----------------------------------|
| Lyon | 0.27 | **0.38** | ~130 euros/nuit |
| Paris | 0.23 | **0.33** | ~306 euros/nuit |
| Bordeaux | 0.51 | **0.62** | ~115 euros/nuit |

La regression multiple ameliore systematiquement tous les indicateurs.
Bordeaux a le meilleur R2 (0.62).

**Pistes d'amelioration :** Random Forest, XGBoost, NLP sur les descriptions, saisonnalite.